# 4. The output head, and the two mixing rules

PARNET predicts three tracks per task. `target` is the RBP eCLIP signal, `control` is the
size-matched input background, and `total` is a learned combination of the two. A fourth
output, `mix_coeff`, is the coefficient that does the combining.

There are **two different combination rules** in the codebase, and they are not
interchangeable. An earlier version of this walkthrough argued that they coincide when
the coefficient is bounded. That argument is wrong, and showing why it is wrong is the
clearest way to understand the head.

In [1]:
import torch

torch.set_num_threads(2)

import functools
import parnet_tutorials as pt
from parnet import layers
from parnet.models import RBPNet

## Part A. What the head actually computes

No layer in PARNET changes sequence length. Everything is `stride=1, padding='same'`, and
the head is pointwise (`kernel_size=1`). So the shape story is simple:

```
embedding            (B, body_width, L)
  |-- head_target    pointwise conv  -> target_logit   (B, tasks, L)
  |-- head_control   pointwise conv  -> control_logit  (B, tasks, L)
  \-- mix_coeff      pool + dense    -> mix_coeff      (B, tasks)
```

Note the last one: `mix_coeff` has no length axis. Both mixing-coefficient layers pool
over the whole window before their linear layers, so there is **one coefficient per
(window, task)**, not one per position. The name `SequenceLinearMix` refers to pooling
over the sequence, not to acting along it.

Each logit track is then converted to a log-density over positions:

```
target_logprob  = target_logit  - logsumexp(target_logit,  dim=position)
control_logprob = control_logit - logsumexp(control_logit, dim=position)
```

After exponentiating, each track sums to 1 across positions. The model predicts signal
**shape**, not depth. That is a direct consequence of the training loss being
multinomial.

In [2]:
head = layers.AdditiveMix(
    num_tasks=2,
    mix_coeff_layer=layers.MixCoeffMLP,
    use_maximum_target_control_logprob=True,
)
x = torch.randn(1, 8, 5)
out = head(x)

for key, value in out.items():
    print(f'{key:12s} {tuple(value.shape)}')

print()
print('mix_coeff has no length axis: one coefficient per (window, task)')
probs = out['total'].exp()
print('each total track sums to 1 over positions:', probs.sum(-1).detach().numpy().round(6))

target       (1, 2, 5)
control      (1, 2, 5)
total        (1, 2, 5)
mix_coeff    (1, 2)
target_weight (1, 2)

mix_coeff has no length axis: one coefficient per (window, task)
each total track sums to 1 over positions: [[1. 1.]]


/tmp/ipykernel_70456/2072060077.py:1: DeprecationWarning: parnet.layers.AdditiveMix was renamed to TargetControlMix in 2026-10 and the old name will be removed at 1.0. Behaviour is unchanged. Pickled checkpoints that still name AdditiveMix keep loading; re-export them to drop the alias.
  head = layers.AdditiveMix(
/tmp/ipykernel_70456/2072060077.py:3: DeprecationWarning: parnet.layers.MixCoeffMLP was renamed to PooledProbMixCoeff in 2026-10 and the old name will be removed at 1.0. Behaviour is unchanged. Pickled checkpoints that still name MixCoeffMLP keep loading; re-export them to drop the alias.
  mix_coeff_layer=layers.MixCoeffMLP,


## Part B. The two rules

Both rules combine the same two log-densities, and both are in `AdditiveMix`. Which one
runs is decided by `use_maximum_target_control_logprob`.

**Convex combination** (`True`), used by the 3M and 7M models:

$$ p_{\text{total}} = w \cdot p_{\text{target}} + (1-w) \cdot p_{\text{control}} $$

This is a weighted average of probabilities, so it needs $w \in [0,1]$. Feed it anything
else and $(1-w)$ goes negative and the logarithm receives a negative argument.

**Log-space addition** (`False`, the default), used by the released 21M:

$$ \log p_{\text{total}} \propto \operatorname{logsumexp}\big(w + \log p_{\text{target}},\ \log p_{\text{control}}\big) $$

Here $w$ is an unbounded logit added before a normalisation, so any real value is legal.

The two layers that produce $w$ advertise which they emit:

In [3]:
for cls in (layers.SequenceLinearMix, layers.MixCoeffMLP, layers.NewMixCoeffMLP):
    print(f'{cls.__name__:20s} emits {cls.mix_coeff_range!r}')

PooledLogitMixCoeff  emits 'logit'
PooledProbMixCoeff   emits 'unit'
PooledProbMixCoeffV2 emits 'unit'


/tmp/ipykernel_70456/2679389148.py:1: DeprecationWarning: parnet.layers.SequenceLinearMix was renamed to PooledLogitMixCoeff in 2026-10 and the old name will be removed at 1.0. Behaviour is unchanged. Pickled checkpoints that still name SequenceLinearMix keep loading; re-export them to drop the alias.
  for cls in (layers.SequenceLinearMix, layers.MixCoeffMLP, layers.NewMixCoeffMLP):
/tmp/ipykernel_70456/2679389148.py:1: DeprecationWarning: parnet.layers.MixCoeffMLP was renamed to PooledProbMixCoeff in 2026-10 and the old name will be removed at 1.0. Behaviour is unchanged. Pickled checkpoints that still name MixCoeffMLP keep loading; re-export them to drop the alias.
  for cls in (layers.SequenceLinearMix, layers.MixCoeffMLP, layers.NewMixCoeffMLP):
/tmp/ipykernel_70456/2679389148.py:1: DeprecationWarning: parnet.layers.NewMixCoeffMLP was renamed to PooledProbMixCoeffV2 in 2026-10 and the old name will be removed at 1.0. Behaviour is unchanged. Pickled checkpoints that still name NewM

### The claim that is wrong

`AdditiveMix`'s own default `mix_coeff_layer` is `MixCoeffMLP`, which is sigmoid-bounded,
the same as the 7M. It is tempting to conclude that a bare `AdditiveMix(num_tasks=...)`
therefore behaves like the 7M.

It does not. **The coefficient's range and the combination rule are independent axes.**
The default `use_maximum_target_control_logprob=False` still selects log-space addition,
so a bare `AdditiveMix` is bounded coefficient + log-sum, which is neither released model.

Here is the measurement. Both heads get bitwise identical `head_target`, `head_control`
and `mix_coeff` parameters, so the only thing that can differ is the rule.

In [4]:
torch.manual_seed(0)
B, H, T, L = 1, 32, 2, 9
x = torch.randn(B, H, L)

# strict_mix=False only to get past the compatibility guard, which exists precisely to
# stop anyone building this by accident. See the next cell.
bare = layers.AdditiveMix(num_tasks=T, strict_mix=False)   # class defaults
new = layers.NewAdditiveMix(num_tasks=T)                   # always the convex rule
bare(x); new(x)                                            # materialise lazy parameters
bare.eval(); new.eval()

with torch.no_grad():
    for a, b in zip(bare.head_target.parameters(), new.head_target.parameters()):
        b.copy_(a)
    for a, b in zip(bare.head_control.parameters(), new.head_control.parameters()):
        b.copy_(a)
    for a, b in zip(bare.mix_coeff.parameters(), new.mix_coeff.parameters()):
        b.copy_(a)
    ob, on = bare(x), new(x)

for key in ['target', 'control', 'mix_coeff', 'total']:
    print(f'{key:10s} max |bare - new| = {(ob[key] - on[key]).abs().max().item():.6g}')

target     max |bare - new| = 0
control    max |bare - new| = 0
mix_coeff  max |bare - new| = 0
total      max |bare - new| = 1.09872


/tmp/ipykernel_70456/3812829284.py:7: DeprecationWarning: parnet.layers.AdditiveMix was renamed to TargetControlMix in 2026-10 and the old name will be removed at 1.0. Behaviour is unchanged. Pickled checkpoints that still name AdditiveMix keep loading; re-export them to drop the alias.
  bare = layers.AdditiveMix(num_tasks=T, strict_mix=False)   # class defaults
/home/user/projects/parnet-project/parnet--analyses--interpretation/.pixi/envs/parnet-dev-cu12/lib/python3.10/site-packages/gin/config.py:1582: UserWarning: PooledProbMixCoeff emits mix_coeff in range 'unit', but TargetControlMix with use_maximum_target_control_logprob=False requires 'logit'. A sigmoid output fed to the log-sum rule is squashed twice, confining the effective target weight to (0.5, 0.731). Pass a matching mix_coeff_layer, or strict_mix=False to proceed anyway.
  return fn(*new_args, **new_kwargs)
/tmp/ipykernel_70456/3812829284.py:8: DeprecationWarning: parnet.layers.NewAdditiveMix was renamed to ConvexTargetCo

`target`, `control` and `mix_coeff` are identical to the last bit. `total` is not. The
rules are different functions of the same inputs.

### The sharper consequence

Under the log-sum rule the effective target weight is `sigmoid(mix_coeff)`. Give it a
coefficient that has already been through a sigmoid and you squash twice: the reachable
effective weight is confined to `(sigmoid(0), sigmoid(1)) = (0.5, 0.731)`.

The control track can then never dominate at any position, whatever the network learns.

In [5]:
w = bare.mix_coeff(x).detach()
effective = torch.sigmoid(w)

print(f'mix_coeff (already bounded): [{w.min():.4f}, {w.max():.4f}]')
print(f'effective target weight    : [{effective.min():.4f}, {effective.max():.4f}]')
print()
lo, hi = torch.sigmoid(torch.tensor([0.0, 1.0]))
print(f'reachable range for any bounded coefficient: ({lo:.4f}, {hi:.4f})')
print('the control track can never dominate')

mix_coeff (already bounded): [0.4885, 0.4985]
effective target weight    : [0.6198, 0.6221]

reachable range for any bounded coefficient: (0.5000, 0.7311)
the control track can never dominate


### The guard

Because this is easy to build by accident, and because the failure is silent in one
direction and a `NaN` in the other, the pairing is checked at construction.

In [6]:
try:
    layers.AdditiveMix(num_tasks=2)   # the class defaults, with no strict_mix escape
except layers.IncompatibleMixError as error:
    print('IncompatibleMixError:')
    print(' ', error)

IncompatibleMixError:
  PooledProbMixCoeff emits mix_coeff in range 'unit', but TargetControlMix with use_maximum_target_control_logprob=False requires 'logit'. A sigmoid output fed to the log-sum rule is squashed twice, confining the effective target weight to (0.5, 0.731). Pass a matching mix_coeff_layer, or strict_mix=False to proceed anyway.
  In call to configurable 'TargetControlMix' (<class 'parnet.layers.TargetControlMix'>)


/tmp/ipykernel_70456/3581474253.py:2: DeprecationWarning: parnet.layers.AdditiveMix was renamed to TargetControlMix in 2026-10 and the old name will be removed at 1.0. Behaviour is unchanged. Pickled checkpoints that still name AdditiveMix keep loading; re-export them to drop the alias.
  layers.AdditiveMix(num_tasks=2)   # the class defaults, with no strict_mix escape


The other direction of the same mismatch is what you hit when loading the released 21M
and switching the flag on: its `SequenceLinearMix` emits an unbounded logit, the convex
rule needs `[0,1]`, and the logarithm receives a negative argument wherever the control
track dominates. That surfaces as `ValueError: Logits contain NaN or Inf values`, far
from its cause.

## Part C. The rules in the released models

No toy weights here. Both models below are real, and they sit on opposite sides of both
axes.

In [7]:
import pandas as pd

rows = []
for key in ['parnet-21m-penalty5', 'parnet-3m-pretrain-E2']:
    model = RBPNet.from_pretrained(pt.model_path(key))
    arch = model.parnet_config['architecture']
    rows.append({
        'model': key,
        'mix_coeff layer': arch['mix_coeff_class'],
        'emits': arch['mix_coeff_range'],
        'convex rule': arch['use_maximum_target_control_logprob'],
        'rule': 'convex combination' if arch['use_maximum_target_control_logprob'] else 'log-space addition',
        'mean w': model.parnet_config['reference']['mean_target_weight_w'],
    })

pd.set_option('display.width', 200)
print(pd.DataFrame(rows).to_string(index=False))

/home/user/projects/parnet-project/parnet--analyses--interpretation/.pixi/envs/parnet-dev-cu12/lib/python3.10/site-packages/torch/nn/modules/conv.py:370: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at /pytorch/aten/src/ATen/native/Convolution.cpp:1025.)
  return F.conv1d(


                model     mix_coeff layer emits  convex rule               rule   mean w
  parnet-21m-penalty5 PooledLogitMixCoeff logit        False log-space addition 0.282869
parnet-3m-pretrain-E2  PooledProbMixCoeff  unit         True convex combination 0.538940


Each model pairs a coefficient range with the rule that accepts it. That is the whole
compatibility story.

`mean w` is the measured mean target weight. For the 21M it is 0.283, so the control
track carries most of the prediction, which is what a penalty of 5.0 was for. Use this
model, not the unpenalised one, for anything that demonstrates mixing: the unpenalised
21M sits at mean w 0.929 with a fifth of its coefficients above 0.99.

## What to take away

1. `mix_coeff` is one number per window and task, not per position.
2. Coefficient range and combination rule are independent. Check both.
3. A bounded coefficient under the log-sum rule is squashed twice into (0.5, 0.731).
4. The constructor refuses incompatible pairings, including `AdditiveMix`'s own defaults.